INSTALL & IMPORT

In [50]:
import pandas as pd
import numpy as np
import re
import torch

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    silhouette_score
)

from sklearn.cluster import KMeans

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

CONFIG

In [51]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
MODEL_SAVE_PATH = "./model/emotion_model"
TOKENIZER_SAVE_PATH = "./model/tokenizer"

LOAD DATA

In [52]:
df = pd.read_csv("ulasan_playstore_livin.csv", encoding='latin1', sep=';')
df = df[['content','score']].dropna()
df.head()

,content,score
0,Aplikasinya luar biasa terutama dalam menguji...,1
1,Saya senang ada aplikasi ini karena sangat mem...,5
2,sering update trs,3
3,SERING GANGGUAN WKWKW ada maintenance sampe ti...,1
4,"aplikasi tidak bisa digunakan, PD saat daftar ...",1


PREPROCESSING & CLEANING

In [53]:
def clean_text(text):

    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["content"].apply(clean_text)

LABEL ENCODING

In [55]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

def auto_emotion_label(text):
    text_low = text.lower()
    if not text_low.strip():
        return "netral" # Default for empty text

    # Ensure text is not too long for the model
    result_sentiment = sentiment_model(str(text_low)[:512])[0]['label']

    if result_sentiment == "positive":
        return "senang"
    elif result_sentiment == "negative":
        # More specific negative emotions based on keywords or general negative
        if any(k in text_low for k in ["marah", "kesal", "benci"]):
            return "marah"
        elif any(k in text_low for k in ["kecewa", "gagal", "lambat", "tidak bisa", "buruk"]):
            return "kecewa"
        elif any(k in text_low for k in ["takut", "cemas", "khawatir"]):
            return "takut"
        else: # Default negative if no specific keyword matches
            return "frustrasi"
    return "netral" # Default for neutral sentiment or unhandled cases

df['emotion'] = df['clean_text'].apply(auto_emotion_label)

label_encoder = LabelEncoder()

df["label_encoded"] = label_encoder.fit_transform(df["emotion"])

emotion_classes = list(label_encoder.classes_)

print("Emotion Classes:")
print(emotion_classes)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: w11wo/indonesian-roberta-base-sentiment-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Emotion Classes:
['frustrasi', 'kecewa', 'marah', 'netral', 'senang']


SPLIT DATA

In [56]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_encoded"]
)


LOAD INDOBERT

In [57]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_)
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TOKENIZATION

In [58]:
def tokenize(batch):

    return tokenizer(
        batch["clean_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/799 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

REMOVE COLUMNS

In [59]:
train_dataset = train_dataset.remove_columns([
    "content",
    "score",
    "clean_text",
    "__index_level_0__"
])

val_dataset = val_dataset.remove_columns([
    "content",
    "score",
    "clean_text",
    "__index_level_0__"
])

train_dataset = train_dataset.rename_column(
    "label_encoded",
    "labels"
)

val_dataset = val_dataset.rename_column(
    "label_encoded",
    "labels"
)

train_dataset.set_format("torch")

val_dataset.set_format("torch")

METRICS

In [60]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

TRAINING

In [61]:
training_args = TrainingArguments(
    output_dir="./model/emotion_model",
    # evaluation_strategy="epoch", # Removed due to potential transformers version incompatibility
    # save_strategy="epoch",       # Removed due to potential transformers version incompatibility
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50
    # load_best_model_at_end=True # Removed as it depends on evaluation_strategy/save_strategy
)

TRAINER

In [62]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)
trainer.train()

Step,Training Loss
50,0.887968
100,0.660966
150,0.391861
200,0.306679
250,0.147831
300,0.127699
350,0.029385
400,0.026259
450,0.023459
500,0.019341


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.26214490056037903, metrics={'train_runtime': 178.3991, 'train_samples_per_second': 22.394, 'train_steps_per_second': 2.803, 'total_flos': 262789244785920.0, 'train_loss': 0.26214490056037903, 'epoch': 5.0})

SAVE MODEL

In [64]:
# Save the model and tokenizer after training
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(TOKENIZER_SAVE_PATH)

# Now load the saved tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_SAVE_PATH
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_SAVE_PATH
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

IMPLICIT SARCASM HANDLING

In [65]:
positive_words = [
    "bagus",
    "mantap",
    "hebat",
    "keren",
    "terbaik"
]

negative_words = [
    "error",
    "gagal",
    "lemot",
    "maintenance",
    "pending"
]

def sarcasm_boost(text, emotion):

    pos_found = any(
        word in text for word in positive_words
    )

    neg_found = any(
        word in text for word in negative_words
    )

    if pos_found and neg_found:

        if emotion == "senang":
            return "frustrasi"

    return emotion

PREDICTION FUNCTION

In [66]:
def predict_emotion(text):

    cleaned = clean_text(text)

    inputs = tokenizer(
        cleaned,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():

        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    prediction = torch.argmax(probs, dim=1).item()
    confidence = probs[0][prediction].item()
    emotion = emotion_classes[prediction]

    # implicit sarcasm handling
    emotion = sarcasm_boost(cleaned, emotion)

    return {
        "text": text,
        "emotion": emotion,
        "confidence": round(confidence * 100, 2),
        "probabilities": probs.numpy()[0]
    }

TESTING

In [67]:
samples = [
    "Bagus banget aplikasinya transfer gagal terus",
    "Login sangat cepat",
    "Mantap maintenance tiap malam",
    "Saya takut saldo hilang",
    "Aplikasi sangat membantu"
]

results = []

for text in samples:

    result = predict_emotion(text)

    print(result)

    row = {
        "text": text,
        "emotion": result["emotion"],
        "confidence": result["confidence"]
    }

    # save probability
    for idx, emotion_name in enumerate(emotion_classes):

        row[emotion_name] = result["probabilities"][idx]

    results.append(row)

{'text': 'Bagus banget aplikasinya transfer gagal terus', 'emotion': 'kecewa', 'confidence': 99.27, 'probabilities': array([5.4502022e-04, 9.9268299e-01, 3.9365338e-03, 7.6296553e-04,
       2.0726118e-03], dtype=float32)}
{'text': 'Login sangat cepat', 'emotion': 'senang', 'confidence': 99.75, 'probabilities': array([5.9826643e-04, 7.2654552e-04, 5.9798331e-04, 5.8873912e-04,
       9.9748850e-01], dtype=float32)}
{'text': 'Mantap maintenance tiap malam', 'emotion': 'frustrasi', 'confidence': 99.78, 'probabilities': array([2.8703953e-04, 4.7518310e-04, 5.1544228e-04, 9.2975708e-04,
       9.9779260e-01], dtype=float32)}
{'text': 'Saya takut saldo hilang', 'emotion': 'frustrasi', 'confidence': 99.45, 'probabilities': array([9.9449557e-01, 2.9323702e-03, 1.0022629e-03, 1.2958061e-03,
       2.7393069e-04], dtype=float32)}
{'text': 'Aplikasi sangat membantu', 'emotion': 'senang', 'confidence': 99.79, 'probabilities': array([1.9942634e-04, 6.6830486e-04, 7.4085413e-04, 5.2948605e-04,
    

SAVE PREDICTION

In [68]:
hasil_df = pd.DataFrame(results)

hasil_df.to_csv(
    "hasil_prediksi.csv",
    index=False
)

print("HASIL PREDIKSI DISIMPAN")

HASIL PREDIKSI DISIMPAN


CLUSTERING

In [69]:
features = hasil_df[emotion_classes]

SCALING

In [71]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)

KMEANS

In [72]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

clusters = kmeans.fit_predict(
    scaled_features
)

hasil_df["cluster"] = clusters

EVALUATION

In [73]:
score = silhouette_score(
    scaled_features,
    clusters
)

print("Silhouette Score:", score)

Silhouette Score: 0.44611207


SAVE CLUSTER

In [74]:
hasil_df.to_csv(
    "hasil_cluster.csv",
    index=False
)

print("CLUSTERING SELESAI")

CLUSTERING SELESAI


In [75]:
from huggingface_hub import notebook_login
notebook_login()

In [76]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id="envidevelopment/model2",
    repo_type="model",
    exist_ok=True
)

RepoUrl('https://huggingface.co/envidevelopment/model2', endpoint='https://huggingface.co', repo_type='model', repo_id='envidevelopment/model2')

In [77]:
api.upload_folder(
    folder_path="model",
    repo_id="envidevelopment/model2",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ckpoint-171/rng_state.pth: 100%|##########| 14.6kB / 14.6kB            

  ...eckpoint-171/scheduler.pt: 100%|##########| 1.47kB / 1.47kB            

  ...int-171/training_args.bin: 100%|##########| 5.14kB / 5.14kB            

...

  ...eckpoint-500/optimizer.pt:   0%|          |  182kB /  996MB            

  ...ckpoint-500/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...n_model/model.safetensors:   0%|          |  552kB /  498MB            

  ...int-500/model.safetensors:   3%|2         | 14.2MB /  498MB            

CommitInfo(commit_url='https://huggingface.co/envidevelopment/model2/commit/37c732c05a5d70b943bf89839a7eec1d144754eb', commit_message='Upload folder using huggingface_hub', commit_description='', oid='37c732c05a5d70b943bf89839a7eec1d144754eb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/envidevelopment/model2', endpoint='https://huggingface.co', repo_type='model', repo_id='envidevelopment/model2'), pr_revision=None, pr_num=None)

In [78]:
from huggingface_hub import HfApi

api = HfApi()

# 1. buat repo dulu
api.create_repo(
    repo_id="envidevelopment/model2",
    repo_type="model",
    exist_ok=True
)

# 2. upload model
api.upload_folder(
    folder_path="model",
    repo_id="envidevelopment/model2",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n_model/training_args.bin: 100%|##########| 5.14kB / 5.14kB            

  ...ckpoint-500/rng_state.pth: 100%|##########| 14.6kB / 14.6kB            

  ...eckpoint-500/scheduler.pt: 100%|##########| 1.47kB / 1.47kB            

...

  ...int-500/model.safetensors:   8%|8         | 40.0MB /  498MB            

  ...int-171/model.safetensors:   6%|6         | 32.0MB /  498MB            

  ...n_model/model.safetensors:   5%|4         | 23.9MB /  498MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/envidevelopment/model2/commit/37c732c05a5d70b943bf89839a7eec1d144754eb', commit_message='Upload folder using huggingface_hub', commit_description='', oid='37c732c05a5d70b943bf89839a7eec1d144754eb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/envidevelopment/model2', endpoint='https://huggingface.co', repo_type='model', repo_id='envidevelopment/model2'), pr_revision=None, pr_num=None)